<a href="https://colab.research.google.com/github/marinhotechdev-ia/AlgoritimosGeneticos/blob/main/alg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmo Genético - Aula 04
Este arquivo foi preparado para ser importado e executado diretamente no **Google Colab**.

In [5]:
import random

# ============================================================
# Parâmetros do problema
# ============================================================
NUM_BITS = 5          # Número de bits do cromossomo (2^5 = 32 > 31)
POP_SIZE = 10         # Tamanho da população
X_MIN = 0             # Valor mínimo de x
X_MAX = 31            # Valor máximo de x

# ============================================================
# 1. Geração de cromossomo aleatório
# ============================================================
def gerar_cromossomo(num_bits=NUM_BITS):
    """
    Gera um cromossomo aleatório (lista de bits 0/1).
    """
    return [random.randint(0, 1) for _ in range(num_bits)]

# ============================================================
# 2. Decodificação: cromossomo binário -> valor inteiro x
# ============================================================
def decodificar(cromossomo):
    """
    Converte um cromossomo binário em um valor inteiro x.
    """
    x = 0
    for i, gene in enumerate(reversed(cromossomo)):
        x += gene * (2 ** i)
    return x

# ============================================================
# 3. Função de aptidão (fitness)
# ============================================================
def aptidao(cromossomo):
    """
    Calcula a aptidão de um cromossomo.
    Problema: maximizar f(x) = x²
    """
    x = decodificar(cromossomo)
    return x ** 2

# ============================================================
# 4. Geração da população inicial
# ============================================================
def gerar_populacao(tamanho=POP_SIZE, num_bits=NUM_BITS):
    """
    Gera a população inicial aleatoriamente.
    """
    return [gerar_cromossomo(num_bits) for _ in range(tamanho)]

# ============================================================
# 5. Avaliação da população
# ============================================================
def avaliar_populacao(populacao):
    """
    Avalia todos os indivíduos da população.
    """
    avaliacao = []
    for i, cromossomo in enumerate(populacao):
        x = decodificar(cromossomo)
        f = aptidao(cromossomo)
        avaliacao.append({
            'indice': i,
            'cromossomo': cromossomo,
            'cromossomo_str': ''.join(str(g) for g in cromossomo),
            'x': x,
            'aptidao': f
        })
    return avaliacao

# ============================================================
# 6. Exibição dos resultados
# ============================================================
def exibir_populacao(avaliacao, geracao=0):
    """
    Exibe a população avaliada em formato de tabela.
    """
    print(f"\n{'='*60}")
    print(f"  GERAÇÃO {geracao}")
    print(f"{'='*60}")
    print(f"  {'Índice':<8} {'Cromossomo':<14} {'x':<6} {'f(x) = x²':<10}")
    print(f"  {'-'*8} {'-'*14} {'-'*6} {'-'*10}")

    for ind in avaliacao:
        print(f"  {ind['indice']:<8} {ind['cromossomo_str']:<14} "
              f"{ind['x']:<6} {ind['aptidao']:<10}")

    # Estatísticas
    aptidoes = [ind['aptidao'] for ind in avaliacao]
    soma = sum(aptidoes)
    media = soma / len(aptidoes)
    melhor = max(aptidoes)
    pior = min(aptidoes)

    print(f"  {'-'*40}")
    print(f"  Soma das aptidões:  {soma}")
    print(f"  Aptidão média:      {media:.2f}")
    print(f"  Melhor aptidão:     {melhor}")
    print(f"  Pior aptidão:       {pior}")
    print(f"{'='*60}\n")

# ============================================================
# 7. Seleção por Roleta
# ============================================================
def selecao_roleta(populacao_avaliada):
    """
    Seleciona um indivíduo da população usando o método da roleta.
    Quanto maior a aptidão, maior a probabilidade de ser selecionado.
    """
    soma_aptidoes = sum(ind['aptidao'] for ind in populacao_avaliada)

    # Caso raro de soma zero (todos os indivíduos com aptidão 0)
    if soma_aptidoes == 0:
        escolhido = random.choice(populacao_avaliada)
        return escolhido['cromossomo'].copy()

    ponto_sorteio = random.uniform(0, soma_aptidoes)
    soma_parcial = 0

    for ind in populacao_avaliada:
        soma_parcial += ind['aptidao']
        if soma_parcial >= ponto_sorteio:
            return ind['cromossomo'].copy()

    # Garantia de retorno
    return populacao_avaliada[-1]['cromossomo'].copy()

# ============================================================
# 8. Cruzamento de 1 ponto
# ============================================================
def cruzamento(pai, mae, taxa_cruzamento=0.7):
    """
    Realiza o cruzamento de um ponto entre dois pais, gerando dois descendentes.
    """
    if random.random() <= taxa_cruzamento:
        ponto = random.randint(1, len(pai) - 1)
        filho1 = pai[:ponto] + mae[ponto:]
        filho2 = mae[:ponto] + pai[ponto:]
        return filho1, filho2
    else:
        # Se não ocorrer cruzamento, retorna cópias idênticas dos pais
        return pai.copy(), mae.copy()

# ============================================================
# 9. Mutação
# ============================================================
def mutacao(cromossomo, taxa_mutacao=0.01):
    """
    Aplica mutação no cromossomo invertendo os bits com uma dada probabilidade.
    """
    novo_cromossomo = []
    for gene in cromossomo:
        if random.random() <= taxa_mutacao:
            novo_cromossomo.append(1 - gene) # 0 vira 1, 1 vira 0
        else:
            novo_cromossomo.append(gene)
    return novo_cromossomo

# ============================================================
# 10. Loop evolutivo
# ============================================================
def evoluir(populacao, num_geracoes=100, taxa_cruzamento=0.7, taxa_mutacao=0.01):
    """
    Executa o ciclo evolutivo por um número definido de gerações.
    """
    melhor_individuo_global = None
    geracao_otimo = None
    historico_aptidao_media = []

    for geracao in range(num_geracoes):
        # Passo 1: Avaliação
        avaliacao = avaliar_populacao(populacao)

        # Passo 2: Exibir geração atual
        exibir_populacao(avaliacao, geracao=geracao)

        # Coleta de métricas e registro do melhor indivíduo
        aptidoes = [ind['aptidao'] for ind in avaliacao]
        media_geracao = sum(aptidoes) / len(aptidoes)
        historico_aptidao_media.append(media_geracao)

        for ind in avaliacao:
            # Atualiza o melhor indivíduo já encontrado em toda a execução
            if melhor_individuo_global is None or ind['aptidao'] > melhor_individuo_global['aptidao']:
                melhor_individuo_global = ind.copy()

            # Registra se encontrou o ótimo global f(x) = 961 para x = 31 (bits: 11111)
            if ind['aptidao'] == 961 and geracao_otimo is None:
                geracao_otimo = geracao

        # Passo 3: Criação da nova população
        nova_populacao = []

        # Gera novos indivíduos até preencher a população
        while len(nova_populacao) < len(populacao):
            # Seleção
            pai = selecao_roleta(avaliacao)
            mae = selecao_roleta(avaliacao)

            # Cruzamento
            filho1, filho2 = cruzamento(pai, mae, taxa_cruzamento)

            # Mutação
            filho1 = mutacao(filho1, taxa_mutacao)
            filho2 = mutacao(filho2, taxa_mutacao)

            nova_populacao.append(filho1)
            if len(nova_populacao) < len(populacao):
                nova_populacao.append(filho2)

        # Atualiza a população para o próximo ciclo
        populacao = nova_populacao

    # Retorna o resultado final acompanhado do registro do melhor indivíduo
    return populacao, melhor_individuo_global, geracao_otimo, historico_aptidao_media

# ============================================================
# Execução obrigatória
# ============================================================


## Execução do Algoritmo
Abaixo, rodamos o algoritmo genético e visualizamos os resultados.

In [6]:
# Semente para permitir a reprodução dos resultados
random.seed(42)

# Inicialização da população
populacao = gerar_populacao(POP_SIZE, NUM_BITS)

# Evolução
populacao_final, melhor, geracao_otimo, historico_media = evoluir(
    populacao,
    num_geracoes=100,
    taxa_cruzamento=0.7,
    taxa_mutacao=0.01
)

# Exibição dos resultados obrigatórios
print("=" * 60)
print("  RESULTADOS OBRIGATÓRIOS")
print("=" * 60)
print("\nMelhor indivíduo encontrado:")
print(f"Cromossomo: {melhor['cromossomo']}")
print(f"Valor de x: {melhor['x']}")
print(f"Aptidão f(x): {melhor['aptidao']}")

if geracao_otimo is not None:
    print(f"Geração em que o ótimo foi alcançado: {geracao_otimo}")
else:
    print("Geração em que o ótimo foi alcançado: O ótimo global (x=31) não foi alcançado nas 100 gerações.")

print("\nAnálise sobre o comportamento da aptidão média:")

# Análise simples baseada na diferença do início e fim
inicio = historico_media[0]
fim = historico_media[-1]

tendencia = "aumentou" if fim > inicio else ("diminuiu" if fim < inicio else "permaneceu estável")

print(f"A aptidão média {tendencia} ao longo das 100 gerações. "
      f"Ela iniciou em {inicio:.2f} na Geração 0 e atingiu {fim:.2f} na Geração 99.")
print("Justificativa: Com o decorrer das gerações, os indivíduos com maior aptidão são mais "
      "selecionados pela Roleta. Como consequência, ocorre a reprodução (cruzamento) "
      "desses bons genes, gerando descendentes melhores, elevando progressivamente a "
      "qualidade geral da população (a aptidão média) até que haja convergência.")



  GERAÇÃO 0
  Índice   Cromossomo     x      f(x) = x² 
  -------- -------------- ------ ----------
  0        00100          4      16        
  1        00010          2      4         
  2        00000          0      0         
  3        01011          11     121       
  4        00111          7      49        
  5        00100          4      16        
  6        10111          23     529       
  7        01010          10     100       
  8        11000          24     576       
  9        01000          8      64        
  ----------------------------------------
  Soma das aptidões:  1475
  Aptidão média:      147.50
  Melhor aptidão:     576
  Pior aptidão:       0


  GERAÇÃO 1
  Índice   Cromossomo     x      f(x) = x² 
  -------- -------------- ------ ----------
  0        10111          23     529       
  1        10111          23     529       
  2        10000          16     256       
  3        11111          31     961       
  4        10110          22    